# Baseline conditions — corrected notebook

Rewrite of `suryar1 (2).ipynb`, addressing the defects in
`SURYA_BASELINE_NOTEBOOK_REVIEW.md`. Every fix is tagged with its review ID
(**B1–B6** blockers, **C1–C7** correctness, **E1–E4** environment) at the point
where it applies.

## What was wrong, in one table

| ID | Defect | Fix here |
|---|---|---|
| B1 | Ran on 5 questions from `train`; `raw_dataset['test']` never used | §3 iterates the full test split, with an assertion on record count |
| B2 | Prompt ended mid-instruction → model completed the instruction | §5 chat template / few-shot completion + stop criteria |
| B3 | `is_correct` was a substring test | §6 `eval_pipeline.exact_match` |
| B4 | Loaded the contaminated test file | §3 requires `_cleaned`, loud failure otherwise |
| B5 | `evaluate_on_custom_benchmark` called but never defined | §9 defines the runner it calls |
| B6 | `nextvalueup/Meta-Llama-3.1-8B` (wrong version, unvetted mirror) | §4 `MODEL_ID` pinned + documented |
| C1 | Non-canonical output schema | §7 `write_predictions` emits toolkit-globbable files |
| C2 | `correct` could be the string `'N/A'` → read as **correct** | §6 always a real bool |
| C3 | Temperature scaling applied as a decoding temp | §8 post-hoc on hedge logits, fitted on val |
| C4 | Oracle had perfect answers + always `[CONFIDENT]` | §7 gold *hedge* over the model's own answers |
| C5 | Self-consistency called RAG internally | §8 samples `plain_generate` |
| C6 | Majority vote over raw 256-token strings | §6 `normalize_for_vote` before counting |
| C7 | RAG corpus embedded the gold answer, built from `train` | §8 statement-form corpus, test questions excluded |
| E1 | Ran on CPU (TPU runtime + bitsandbytes) → `KeyboardInterrupt` | §1 hard GPU assert |
| E2 | `change_year` taken from `train`'s `t_end` | §7 from the test record |
| E3/E4 | Dead training scaffolding, stale `corpus.txt`/Mistral markdown | removed |

## Before you run

1. **Runtime → Change runtime type → GPU** (A100 or L4). Not TPU — bitsandbytes
   has no TPU path, and that is what killed the previous run.
2. §10 self-tests every non-GPU helper. Run it first; it takes seconds and
   catches most breakage before you spend GPU time.
3. §11 is the verification gate. **Do not send results that fail it.**

## Open item that is not Surya's to fix

`temporal_delta_test_cleaned.jsonl` is **not published** in
`jasontae/temporal-delta` (review B4). §3 fails loudly if it is missing rather
than silently falling back to the contaminated file. If it still isn't there,
that's on the eval side — ping Jason before running.

## 1. Environment — fail fast (E1)

In [ ]:
# FIX E1: the previous run was on CPU with a 4-bit quantised 8B model and died
# in bitsandbytes' forward pass. Refuse to continue rather than burn an hour.
import torch, sys

if not torch.cuda.is_available():
    raise SystemExit(
        "No CUDA device.\n"
        "Runtime -> Change runtime type -> Hardware accelerator: GPU (A100/L4).\n"
        "TPU will not work: bitsandbytes 4-bit has no TPU backend."
    )

DEVICE = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
%pip install -qqq -U "transformers>=4.44" accelerate bitsandbytes sentence-transformers datasets huggingface_hub

## 2. Metrics — one source of truth

FIX B3/C2/C1: import the project's real metric code instead of reimplementing it. Divergent local copies of `is_correct` are how the substring bug survived three reviews — and the same defect exists in David's inference output (see `TCL_V6_DIAGNOSIS_AND_PROPOSAL.md` §1.6).

In [ ]:
import os, sys, subprocess

REPO = "https://github.com/jasontae/temporal-self-consistency.git"
if not os.path.isdir("/content/temporal-self-consistency"):
    subprocess.run(["git", "clone", "-q", REPO, "/content/temporal-self-consistency"], check=True)

for p in ("/content/temporal-self-consistency/src/evaluation",
          "/content/temporal-self-consistency"):
    if p not in sys.path:
        sys.path.insert(0, p)

try:
    from eval_pipeline import (exact_match, token_f1, normalize_answer,
                               compute_ece, HEDGE_TO_CONFIDENCE)
except ImportError as e:
    raise SystemExit(
        f"Could not import eval_pipeline ({e}).\n"
        "Upload eval_toolkit/eval_pipeline.py to /content and re-run.\n"
        "Do NOT reimplement exact_match locally -- that is review defect B3."
    )

print("eval_pipeline loaded. Hedge->confidence:", HEDGE_TO_CONFIDENCE)

## 3. Data — full **cleaned** test split (B1, B4)

In [ ]:
# FIX B4: DATASET.md mandates the decontaminated test set (3,456 records).
# The 3,622-record original leaks train/test overlap.
# FIX B1: the previous notebook evaluated 5 questions from *train*. We load the
# test split and assert on it.
from huggingface_hub import snapshot_download
from datasets import load_dataset

DATA_DIR = "./data/prep"
snapshot_download(repo_id="jasontae/temporal-delta", repo_type="dataset",
                  local_dir=DATA_DIR)
print("files:", sorted(os.listdir(DATA_DIR)))

CLEANED = os.path.join(DATA_DIR, "temporal_delta_test_cleaned.jsonl")
ORIGINAL = os.path.join(DATA_DIR, "temporal_delta_test.jsonl")

if not os.path.exists(CLEANED):
    raise SystemExit(
        "temporal_delta_test_cleaned.jsonl is NOT in the dataset repo.\n"
        "This is review blocker B4 and it is an eval-side task, not yours.\n"
        "Ping Jason to publish it. Do not silently fall back to\n"
        f"{os.path.basename(ORIGINAL)} -- that set is contaminated (DATASET.md)."
    )

raw = load_dataset("json", data_files={
    "train": os.path.join(DATA_DIR, "temporal_delta_train.jsonl"),
    "validation": os.path.join(DATA_DIR, "temporal_delta_val.jsonl"),
    "test": CLEANED,
})

EXPECTED_TEST_N = 3456
n_test = len(raw["test"])
print(f"test records: {n_test} (expected {EXPECTED_TEST_N})")
assert n_test == EXPECTED_TEST_N, (
    f"Test split has {n_test} records, expected {EXPECTED_TEST_N}. "
    "Wrong file or stale dataset version -- stop and check DATASET.md."
)

# Note (review D1): the published train split is 14,206 records, while
# DATASET.md describes 67,600 (5.8x deploy-offset augmentation). Unresolved.
print(f"train records: {len(raw['train'])}  (DATASET.md says 67,600 -- see review D1)")
print("example test record:", {k: raw["test"][0][k]
                               for k in ("question", "answer", "volatility")})

## 4. Model (B6)

In [ ]:
# FIX B6: previous notebook used "nextvalueup/Meta-Llama-3.1-8B" -- LLaMA-3.1,
# not the LLaMA-3 8B David trained on, and from an unaffiliated mirror.
# BASELINES_STATUS.md requires matching David's base model for a fair comparison.
#
# DECISION NEEDED: the -Instruct variant is what makes a chat template (and
# therefore a non-echoing prompt) available. If David's SFT started from the
# non-Instruct base, either move all baselines to Instruct or use the few-shot
# completion path in section 5 -- but be consistent and state it in the paper.
MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.bfloat16,
                         bnb_4bit_use_double_quant=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, torch_dtype=torch.bfloat16,
    device_map={"": 0})
model.eval()

HAS_CHAT_TEMPLATE = getattr(tokenizer, "chat_template", None) is not None
print(f"loaded {MODEL_ID} | chat_template={HAS_CHAT_TEMPLATE}")

## 5. Prompting — the fix for prompt echo (B2)

The previous prompt ended **inside** the instruction:

```python
full_query = query + "\n\nPlease classify your confidence ... [UNKNOWN]"
```

so the model's continuation was the rest of the instruction. Every prediction
began `"to your answer."`, `"or [UNCERTAIN]."`, `"to the end of your answer."`.

Two corrections: put the instruction where it can't be completed (system turn,
or few-shot demonstrations), and **stop generation** before the model invents a
new `Question:` — which is what produced those run-on outputs.

In [ ]:
from transformers import StoppingCriteria, StoppingCriteriaList

SYSTEM = ("Answer the question as briefly as possible -- just the answer, no "
          "explanation. Then append exactly one confidence token: "
          "[CONFIDENT], [COND_CONFIDENT], [TEMPORAL_HEDGE], or [UNKNOWN].")

FEWSHOT = [
    ("What is the chemical formula of water?", "H2O [CONFIDENT]"),
    ("What is the capital of France?",         "Paris [COND_CONFIDENT]"),
    ("Who is the CEO of Twitter?",             "Linda Yaccarino [TEMPORAL_HEDGE]"),
]


def build_prompt(question, context=None):
    """Chat template when available, else few-shot completion.

    FIX B2: in both branches the prompt ends where an ANSWER is the natural
    continuation, never mid-instruction.
    """
    user = f"Question: {question}"
    if context:
        user = f"Context:\n{context}\n\n{user}"

    if HAS_CHAT_TEMPLATE:
        msgs = [{"role": "system", "content": SYSTEM}]
        for q, a in FEWSHOT:
            msgs += [{"role": "user", "content": f"Question: {q}"},
                     {"role": "assistant", "content": a}]
        msgs.append({"role": "user", "content": user})
        return tokenizer.apply_chat_template(msgs, tokenize=False,
                                             add_generation_prompt=True)

    demos = "\n\n".join(f"Question: {q}\nAnswer: {a}" for q, a in FEWSHOT)
    head = f"Context:\n{context}\n\n" if context else ""
    return f"{SYSTEM}\n\n{demos}\n\n{head}Question: {question}\nAnswer:"


class StopOnStrings(StoppingCriteria):
    """FIX B2 (run-on): halt before the model fabricates a new Q/A pair."""
    def __init__(self, tok, stops, prompt_len):
        self.tok, self.stops, self.prompt_len = tok, stops, prompt_len

    def __call__(self, input_ids, scores, **kw):
        tail = self.tok.decode(input_ids[0][self.prompt_len:],
                               skip_special_tokens=True)
        return any(s in tail for s in self.stops)


STOP_STRINGS = ["\nQuestion:", "\n\nQuestion:", "Question:", "\nContext:"]


@torch.inference_mode()
def generate_raw(prompt, max_new_tokens=48, temperature=0.7, do_sample=True):
    enc = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    plen = enc["input_ids"].shape[1]
    out = model.generate(
        **enc, max_new_tokens=max_new_tokens, do_sample=do_sample,
        temperature=temperature if do_sample else None,
        pad_token_id=tokenizer.eos_token_id,
        stopping_criteria=StoppingCriteriaList(
            [StopOnStrings(tokenizer, STOP_STRINGS, plen)]),
    )
    text = tokenizer.decode(out[0][plen:], skip_special_tokens=True)
    for s in STOP_STRINGS:                       # trim anything past the stop
        if s in text:
            text = text.split(s)[0]
    return text.strip()

## 6. Parsing and scoring (B2-detection, B3, C2, C6)

In [ ]:
# === Hedge parsing =========================================================
# FIX B2/parse: unchanged logic from the original (it was correct), but a
# parse failure is now RECORDED instead of silently becoming [UNKNOWN].
# Silently defaulting to [UNKNOWN] is what hid the prompt-echo bug: every
# broken generation looked like a confident-refusal rather than a parse error.
import re

HEDGE_TOKENS = ["[CONFIDENT]", "[COND_CONFIDENT]", "[TEMPORAL_HEDGE]", "[UNKNOWN]"]
_HEDGE_RE = re.compile(
    r"(\[CONFIDENT\]|\[COND_CONFIDENT\]|\[TEMPORAL_HEDGE\]|\[UNKNOWN\])\s*$"
)

# Instruction fragments that indicate the model completed the prompt instead
# of answering it. Seen verbatim in the previous run's outputs.
_ECHO_PREFIXES = (
    "to your answer", "to the end of your answer", "to the answer",
    "or [", "and [", ". the tokens are", "please classify",
)


def extract_hedge_and_answer(text):
    """Return (answer, hedge, parse_failed, echo_detected)."""
    t = (text or "").strip()
    m = _HEDGE_RE.search(t)
    if m:
        hedge = m.group(1)
        answer = t[:m.start()].strip()
        parse_failed = False
    else:
        hedge = "[UNKNOWN]"
        answer = t
        parse_failed = True

    low = answer.lower().lstrip()
    echo = any(low.startswith(p) for p in _ECHO_PREFIXES)
    return answer, hedge, parse_failed, echo

In [ ]:
# === Scoring ===============================================================
# FIX B3/C2: correctness comes from eval_pipeline.exact_match, never a
# substring test, and is always a real bool. The old rule scored
#   pred="or [DONT_KNOW]. The CEO of Twitter is Jack Dorsey..." vs gold="Jack Dorsey"
# as CORRECT. There is now a regression test for exactly that case.

def score_record(predicted_answer, gold_answer):
    """Strict EM + token F1. Returns real bools/floats, never 'N/A'."""
    if gold_answer is None or predicted_answer is None:
        return {"correct": False, "em": 0.0, "f1": 0.0, "gold_missing": True}
    return {
        "correct": bool(exact_match(predicted_answer, gold_answer)),
        "em": 1.0 if exact_match(predicted_answer, gold_answer) else 0.0,
        "f1": float(token_f1(predicted_answer, gold_answer)),
        "gold_missing": False,
    }


def normalize_for_vote(text):
    """Short-span normalisation used before majority voting.

    FIX C6: voting on raw 256-token samples never finds agreement, so
    Counter.most_common returned the first sample rather than a consensus.
    """
    t = normalize_answer(text or "")
    return " ".join(t.split()[:8])

## 7. Canonical output and the Oracle (C1, C4)

In [ ]:
# === Canonical output ======================================================
# FIX C1: the eval toolkit globs
#   <exp>_<cond>_seed<N>_<benchmark>_predictions.jsonl
# and reads 'question', not 'query'. One flat file with a 'model_type' column
# is invisible to run_results.py / per_benchmark_breakdown.py.
import json
import os

CONDITION_SLUGS = {
    "base_llm":            "exp1_basellm",
    "self_consistency":    "exp4_selfcons",
    "temperature_scaling": "exp5_tempscale",
    "rag":                 "exp6_rag",
    "oracle":              "exp9_oracle",
}


def make_record(rec, predicted_answer, predicted_hedge, *,
                parse_failed=False, echo_detected=False, confidence=None):
    """Build one canonical prediction record."""
    gold = rec.get("answer")
    s = score_record(predicted_answer, gold)
    out = {
        "question":         rec.get("question"),
        "predicted_answer": predicted_answer,
        "gold_answer":      gold,
        "predicted_hedge":  predicted_hedge,
        "correct":          s["correct"],
        "volatility":       rec.get("volatility"),
        "parse_failed":     bool(parse_failed),
        "echo_detected":    bool(echo_detected),
    }
    ce = rec.get("t_end") or rec.get("validity_end")
    if ce is not None:
        try:
            out["change_year"] = int(ce)
        except (TypeError, ValueError):
            pass
    if confidence is not None:
        out["confidence"] = float(confidence)
    return out


def prediction_path(out_dir, condition, seed, benchmark):
    slug = CONDITION_SLUGS[condition]
    return os.path.join(
        out_dir, f"{slug}_seed{seed}_{benchmark}_predictions.jsonl")


def write_predictions(records, out_dir, condition, seed, benchmark):
    os.makedirs(out_dir, exist_ok=True)
    path = prediction_path(out_dir, condition, seed, benchmark)
    with open(path, "w") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"  wrote {len(records):>5} records -> {os.path.basename(path)}")
    return path

In [ ]:
# === Oracle ================================================================
# FIX C4: BASELINES_STATUS.md defines Oracle as the ALWAYS-CORRECT HEDGE
# ceiling -- the model's own answers paired with the gold hedge for each
# fact's volatility class. The previous version handed it perfect answers AND
# pinned the hedge to [CONFIDENT], which on a ~90%-fast set is the worst
# possible hedge. Its ECE was |1.0 - 0.95| = 0.05 by construction: it measured
# nothing.
GOLD_HEDGE_FOR_VOLATILITY = {
    "immutable": "[CONFIDENT]",
    "slow":      "[COND_CONFIDENT]",
    "fast":      "[TEMPORAL_HEDGE]",
}


def build_oracle_records(base_records):
    """Take base-LLM predictions; replace only the hedge with the gold hedge."""
    out = []
    for r in base_records:
        o = dict(r)
        o["predicted_hedge"] = GOLD_HEDGE_FOR_VOLATILITY.get(
            r.get("volatility"), "[TEMPORAL_HEDGE]")
        out.append(o)
    return out

## 8. The four inference conditions (C3, C5, C7)

Each is a *separate* condition. The previous notebook confounded them:
self-consistency called RAG internally (C5), and "temperature scaling" was a
decoding parameter rather than post-hoc calibration (C3).

In [ ]:
# --- Base LLM -------------------------------------------------------------
def plain_generate(question, temperature=0.7, do_sample=True):
    raw = generate_raw(build_prompt(question), temperature=temperature,
                       do_sample=do_sample)
    ans, hedge, pf, echo = extract_hedge_and_answer(raw)
    return {"predicted_answer": ans, "predicted_hedge": hedge,
            "parse_failed": pf, "echo_detected": echo}


# --- Self-consistency -----------------------------------------------------
# FIX C5: samples the BASE model, not RAG.
# FIX C6: votes on normalised short spans, not raw 256-token strings.
from collections import Counter


def self_consistency_generate(question, k=5, temperature=0.8):
    answers, hedges, meta = [], [], []
    for _ in range(k):
        r = plain_generate(question, temperature=temperature, do_sample=True)
        answers.append(r["predicted_answer"])
        hedges.append(r["predicted_hedge"])
        meta.append((r["parse_failed"], r["echo_detected"]))

    keys = [normalize_for_vote(a) for a in answers]
    if not keys:
        return {"predicted_answer": "", "predicted_hedge": "[UNKNOWN]",
                "parse_failed": True, "echo_detected": False}
    win = Counter(keys).most_common(1)[0][0]
    idx = keys.index(win)                       # a real representative sample
    return {"predicted_answer": answers[idx],
            "predicted_hedge": Counter(hedges).most_common(1)[0][0],
            "parse_failed": meta[idx][0], "echo_detected": meta[idx][1],
            "agreement": Counter(keys)[win] / len(keys)}

In [ ]:
# --- RAG ------------------------------------------------------------------
# FIX C7: documents are STATEMENTS, not "Question: ... Answer: ..." pairs, so
# retrieval no longer hands the model the gold answer verbatim. Any train
# document whose question matches a test question is dropped.
#
# Design note for the paper: the corpus is built from the TRAIN split
# (2018-2021 facts). That is deliberate -- it simulates a stale knowledge base,
# which is the point of a RAG baseline in a temporal-drift study. State it
# explicitly rather than letting a reader assume a live index.
from sentence_transformers import SentenceTransformer

test_questions = {q.strip() for q in raw["test"]["question"]}

corpus, corpus_meta = [], []
for e in raw["train"]:
    if (e.get("question") or "").strip() in test_questions:
        continue                                 # no test-question leakage
    ent, rel = e.get("entity_label"), e.get("relation") or e.get("property")
    val, t0, t1 = e.get("value_label"), e.get("t_start"), e.get("t_end")
    if not (ent and val):
        continue
    if "http" in str(val):                       # review D2: residual URI leakage
        continue
    corpus.append(f"From {t0} to {t1}, the {rel} of {ent} was {val}.")
    corpus_meta.append(e.get("entity_id"))

print(f"corpus: {len(corpus)} statement documents "
      f"({len(raw['train']) - len(corpus)} dropped: test-overlap / URI / incomplete)")

embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")
corpus_emb = embedder.encode(corpus, convert_to_tensor=True,
                             normalize_embeddings=True, batch_size=256,
                             show_progress_bar=True)


@torch.inference_mode()
def retrieve(query, top_k=3):
    q = embedder.encode(query, convert_to_tensor=True, normalize_embeddings=True)
    sims = torch.mv(corpus_emb, q)               # stays on GPU, unlike the old version
    idx = torch.topk(sims, k=min(top_k, len(corpus))).indices.tolist()
    return [corpus[i] for i in idx]


def rag_generate(question, temperature=0.7):
    ctx = "\n".join(retrieve(question))
    raw_out = generate_raw(build_prompt(question, context=ctx),
                           temperature=temperature)
    ans, hedge, pf, echo = extract_hedge_and_answer(raw_out)
    return {"predicted_answer": ans, "predicted_hedge": hedge,
            "parse_failed": pf, "echo_detected": echo}

In [ ]:
# === Temperature scaling (post-hoc) ========================================
# FIX C3: this is a CALIBRATION method, not a decoding parameter. The previous
# notebook passed T=0.2616 to model.generate(), which changes what the model
# SAYS and leaves confidence uncalibrated -- a different intervention with the
# same name.
#
# Correct form: take the model's logits over the four hedge continuations,
# divide by T, softmax. The generated answer is untouched; only the confidence
# estimate changes. T must be FIT on the validation split, not assumed.
import numpy as np


def apply_temperature(hedge_logits, T):
    """hedge_logits: array (4,) over HEDGE_TOKENS. Returns calibrated probs."""
    z = np.asarray(hedge_logits, dtype=np.float64) / float(T)
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()


def fit_temperature(val_logits, val_correct, grid=None):
    """Fit T by minimising NLL of correctness under the hedge-confidence map.

    val_logits:  (N,4) hedge logits on the validation split
    val_correct: (N,)  bools
    """
    if grid is None:
        grid = np.concatenate([np.linspace(0.05, 1.0, 40),
                               np.linspace(1.0, 5.0, 40)])
    conf_vec = np.array([HEDGE_TO_CONFIDENCE[h] for h in HEDGE_TOKENS])
    y = np.asarray(val_correct, dtype=np.float64)
    best_T, best_nll = 1.0, float("inf")
    for T in grid:
        p = np.stack([apply_temperature(z, T) for z in val_logits])
        conf = p @ conf_vec                       # expected confidence
        conf = np.clip(conf, 1e-6, 1 - 1e-6)
        nll = -np.mean(y * np.log(conf) + (1 - y) * np.log(1 - conf))
        if nll < best_nll:
            best_T, best_nll = float(T), float(nll)
    return best_T, best_nll

In [ ]:
# --- Temperature scaling: reading the hedge logits -------------------------
# FIX C3 (continued). The answer is generated ONCE, greedily, and left alone.
# Only the confidence estimate is recalibrated -- that is what makes this
# temperature *scaling* rather than temperature *sampling*.
HEDGE_FIRST_IDS = [tokenizer.encode(h, add_special_tokens=False)[0]
                   for h in HEDGE_TOKENS]


@torch.inference_mode()
def hedge_logits_for(question, answer_text, context=None):
    """Logits over the four hedge tokens at the position right after the answer."""
    prompt = build_prompt(question, context=context) + " " + answer_text.strip() + " "
    enc = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    logits = model(**enc).logits[0, -1, :]
    return logits[HEDGE_FIRST_IDS].float().cpu().numpy()


def temperature_scaled_generate(question, T):
    r = plain_generate(question, do_sample=False)          # greedy, fixed answer
    z = hedge_logits_for(question, r["predicted_answer"])
    p = apply_temperature(z, T)
    conf = float(p @ np.array([HEDGE_TO_CONFIDENCE[h] for h in HEDGE_TOKENS]))
    r["predicted_hedge"] = HEDGE_TOKENS[int(np.argmax(p))]
    r["confidence"] = conf
    return r

## 9. Runner (B1, B5)

FIX B5: `evaluate_on_custom_benchmark` was called four times in the old cell 56
and defined nowhere — the notebook could not reproduce its own results. This is
that function.

FIX B1: iterates the **whole** test split. It checkpoints every `flush_every`
records and resumes from what is already on disk, because 3,456 records ×
several conditions will outlive a Colab session.

In [ ]:
import time


def evaluate_condition(condition, gen_fn, dataset, out_dir="./baseline_preds",
                       seed=42, benchmark="temporal_delta", limit=None,
                       flush_every=100):
    os.makedirs(out_dir, exist_ok=True)
    path = prediction_path(out_dir, condition, seed, benchmark)

    done = []
    if os.path.exists(path):                     # resume
        with open(path) as f:
            done = [json.loads(l) for l in f]
        print(f"[{condition}] resuming: {len(done)} records already on disk")

    records = list(done)
    todo = list(dataset)[len(records): limit if limit else None]
    if not todo:
        print(f"[{condition}] already complete ({len(records)})")
        return records

    t0 = time.time()
    for i, rec in enumerate(todo, start=1):
        try:
            g = gen_fn(rec["question"])
        except Exception as e:                   # never lose a long run
            g = {"predicted_answer": "", "predicted_hedge": "[UNKNOWN]",
                 "parse_failed": True, "echo_detected": False}
            print(f"  !! {type(e).__name__} on record {len(records)}: {e}")
        records.append(make_record(
            rec, g["predicted_answer"], g["predicted_hedge"],
            parse_failed=g.get("parse_failed", False),
            echo_detected=g.get("echo_detected", False),
            confidence=g.get("confidence")))

        if i % flush_every == 0 or i == len(todo):
            with open(path, "w") as f:
                for r in records:
                    f.write(json.dumps(r, ensure_ascii=False) + "\n")
            rate = i / (time.time() - t0)
            eta = (len(todo) - i) / max(rate, 1e-9) / 60
            print(f"  [{condition}] {len(records)}/{len(dataset)} "
                  f"({rate:.1f} rec/s, ETA {eta:.0f} min)")

    print(f"[{condition}] done -> {os.path.basename(path)}")
    return records

In [ ]:
# --- Fit T on the VALIDATION split (C3) -----------------------------------
# T must be fitted, not assumed. Logan's 0.2616 is a starting point to compare
# against, not a value to hard-code.
N_VAL_FIT = 300
val_logits, val_correct = [], []
for rec in list(raw["validation"])[:N_VAL_FIT]:
    r = plain_generate(rec["question"], do_sample=False)
    val_logits.append(hedge_logits_for(rec["question"], r["predicted_answer"]))
    val_correct.append(exact_match(r["predicted_answer"], rec["answer"]))

FITTED_T, nll = fit_temperature(np.array(val_logits), np.array(val_correct))
print(f"fitted T = {FITTED_T:.4f} (val NLL {nll:.4f}) on n={len(val_logits)}")
print(f"Logan's reference T = 0.2616")

In [ ]:
# --- Run all conditions ---------------------------------------------------
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

OUT_DIR = "./baseline_preds"
test = raw["test"]

# Start with LIMIT=25 to sanity-check the whole path, then set LIMIT=None.
LIMIT = 25

base_records = evaluate_condition(
    "base_llm", lambda q: plain_generate(q, do_sample=False),
    test, OUT_DIR, SEED, limit=LIMIT)

sc_records = evaluate_condition(
    "self_consistency", lambda q: self_consistency_generate(q, k=5),
    test, OUT_DIR, SEED, limit=LIMIT)

ts_records = evaluate_condition(
    "temperature_scaling", lambda q: temperature_scaled_generate(q, FITTED_T),
    test, OUT_DIR, SEED, limit=LIMIT)

rag_records = evaluate_condition(
    "rag", lambda q: rag_generate(q), test, OUT_DIR, SEED, limit=LIMIT)

# FIX C4: Oracle = model's own answers + the GOLD hedge. Derived, not generated.
oracle_records = build_oracle_records(base_records)
write_predictions(oracle_records, OUT_DIR, "oracle", SEED, "temporal_delta")

## 10. Self-tests — run this **before** spending GPU time

In [ ]:
# Exercises every helper that does not need the model. All of these assertions
# encode a specific defect from the review.
def _selftest():
    fails = []
    def ck(c, m):
        print(f"  [{'ok  ' if c else 'FAIL'}] {m}")
        if not c: fails.append(m)

    # B2 -- echo detection
    _, h, pf, ec = extract_hedge_and_answer("Jack Dorsey [TEMPORAL_HEDGE]")
    ck((h, pf, ec) == ("[TEMPORAL_HEDGE]", False, False), "clean generation parses")
    _, _, pf, ec = extract_hedge_and_answer("to your answer.")
    ck(pf and ec, "prompt-echo output flagged, not silently [UNKNOWN]")

    # B3 -- the exact false positive from the old notebook
    bad = ("or [DONT_KNOW].\n\nThe CEO of Twitter is Jack Dorsey. "
           "He is the CEO of both Twitter and Square, Inc.")
    ck("jack dorsey" in bad.lower(), "old substring rule would score this correct")
    ck(score_record(bad, "Jack Dorsey")["correct"] is False,
       "exact_match correctly scores it WRONG")

    # C2
    ck(score_record("x", None)["correct"] is False, "missing gold -> False, not 'N/A'")

    # C6
    ck(normalize_for_vote("Jack Dorsey is the CEO.") ==
       normalize_for_vote("jack dorsey is the ceo!!"), "vote keys normalise")

    # C1
    p = prediction_path("/tmp", "base_llm", 42, "temporal_delta")
    ck(p.endswith("seed42_temporal_delta_predictions.jsonl"), "filename is toolkit-globbable")

    # C4
    o = build_oracle_records([make_record(
        {"question": "q", "answer": "A", "volatility": "immutable"}, "WRONG", "[UNKNOWN]")])
    ck(o[0]["predicted_hedge"] == "[CONFIDENT]" and o[0]["correct"] is False,
       "oracle takes the gold hedge but keeps the model's answer")

    # C3
    pr = apply_temperature(np.array([2.0, 1.0, 0.5, 0.0]), 0.2616)
    ck(abs(pr.sum() - 1) < 1e-9 and pr[0] > 0.9, "temperature scaling sharpens and normalises")

    print("\n" + ("SELF-TESTS PASSED" if not fails else f"{len(fails)} FAILED: {fails}"))
    return not fails


assert _selftest(), "fix the failures above before running the model"

## 11. Verification gate + handoff

In [ ]:
# === Verification gate =====================================================
# Run this before sending anything to Jason. Every check corresponds to a
# defect from the review; a run that fails any of them is not a valid baseline.
from collections import Counter


def verification_gate(records, expected_n=None, label=""):
    checks, ok = [], True

    if expected_n is not None:
        p = len(records) == expected_n
        checks.append((p, f"record count == {expected_n} (got {len(records)})"))
        ok &= p

    n_echo = sum(1 for r in records if r.get("echo_detected"))
    p = n_echo == 0
    checks.append((p, f"prompt-echo predictions == 0 (got {n_echo})"))
    ok &= p

    n_fail = sum(1 for r in records if r.get("parse_failed"))
    p = n_fail <= 0.02 * max(1, len(records))
    checks.append((p, f"hedge parse failures <= 2% (got {n_fail})"))
    ok &= p

    bad = {r.get("predicted_hedge") for r in records} - set(HEDGE_TOKENS)
    p = not bad
    checks.append((p, f"hedges within taxonomy (offenders: {sorted(bad) or 'none'})"))
    ok &= p

    p = all(isinstance(r.get("correct"), bool) for r in records)
    checks.append((p, "every 'correct' is a real bool (never 'N/A')"))
    ok &= p

    n_url = sum(1 for r in records
                if "http" in str(r.get("predicted_answer", "")).lower())
    p = n_url == 0
    checks.append((p, f"no URLs in predictions (got {n_url})"))
    ok &= p

    # Collapse must be judged RELATIVE TO THE DATA. temporal_delta is ~90%
    # fast by construction, so heavy [TEMPORAL_HEDGE] there is correct, not
    # collapse. What is pathological is the model being LESS discriminating
    # than the volatility mix warrants -- e.g. v6 seeds 501-505 sat at 99.9%
    # [COND_CONFIDENT] on a mixed set.
    n = max(1, len(records))
    pred_dist = Counter(r.get("predicted_hedge") for r in records)
    gold_dist = Counter(GOLD_HEDGE_FOR_VOLATILITY.get(r.get("volatility"))
                        for r in records)
    pred_top_tok, pred_top_n = pred_dist.most_common(1)[0] if pred_dist else ("-", 0)
    gold_top_tok, gold_top_n = gold_dist.most_common(1)[0] if gold_dist else ("-", 0)
    pred_top, gold_top = pred_top_n / n, gold_top_n / n

    p = not (pred_top >= 0.98 and gold_top < 0.95)
    checks.append((p, f"no hedge collapse (pred top {pred_top_tok} {pred_top:.1%} "
                      f"vs gold top {gold_top_tok} {gold_top:.1%})"))
    ok &= p

    # Cheap over/under-hedging sanity check: on a benchmark dominated by one
    # volatility class the modal hedge should be that class's gold hedge.
    if gold_top >= 0.90:
        p = pred_top_tok == gold_top_tok
        checks.append((p, f"modal hedge matches modal gold "
                          f"({pred_top_tok} vs {gold_top_tok})"))
        ok &= p

    print(f"\n--- verification gate: {label} ---")
    for passed, msg in checks:
        print(f"  [{'PASS' if passed else 'FAIL'}] {msg}")
    print(f"  => {'READY' if ok else 'NOT READY -- do not send'}")
    return ok

In [ ]:
# Gate every condition. Anything that fails is not a valid baseline.
all_ok = True
for cond, recs in (("base_llm", base_records),
                   ("self_consistency", sc_records),
                   ("temperature_scaling", ts_records),
                   ("rag", rag_records),
                   ("oracle", oracle_records)):
    all_ok &= verification_gate(
        recs, expected_n=(LIMIT or len(test)), label=cond)

print("\n" + "=" * 60)
print("OVERALL:", "READY TO SEND" if all_ok else "NOT READY -- see failures above")
print("=" * 60)

In [ ]:
# Local ECE preview, using the project's own metric, so you can see the numbers
# before handing anything over.
print(f"{'condition':<22}{'n':>7}{'ECE':>9}{'EM':>9}{'echo%':>8}{'parsefail%':>12}")
for cond, recs in (("base_llm", base_records), ("self_consistency", sc_records),
                   ("temperature_scaling", ts_records), ("rag", rag_records),
                   ("oracle", oracle_records)):
    if not recs:
        continue
    n = len(recs)
    ece = compute_ece(recs)["ece"]
    em = sum(r["correct"] for r in recs) / n
    echo = 100 * sum(bool(r.get("echo_detected")) for r in recs) / n
    pf = 100 * sum(bool(r.get("parse_failed")) for r in recs) / n
    print(f"{cond:<22}{n:>7}{ece:>9.4f}{em:>9.4f}{echo:>7.1f}%{pf:>11.1f}%")

### Handoff

Once §11 reports **READY** on a full (`LIMIT = None`) run:

```bash
# zip and send ./baseline_preds/ to Jason, then he runs:
python3 per_benchmark_breakdown.py --pred-dir <folder> --out-dir ./results
python3 url_leakage_check.py       --pred-dir <folder>
python3 run_results.py             --pred-dir <folder> --out-dir ./results
```

`run_results.py` treats every non-TSCT condition as a baseline in the Bonferroni
comparison, so the number of conditions here directly sets `n_comparisons` — the
open significance-reporting decision in `docs/EVAL_PIPELINE.md`. Tell Jason how
many conditions you shipped.

### Still out of scope for this notebook

- **Label smoothing** requires fine-tuning (CE + ε=0.1) and cannot be produced by
  a frozen model. It needs its own training run — do not add a training cell
  here (review E3).
- **SFT / TSCT** come from David.
- **SFT+TCL+DPO** needs MTurk preference pairs.
- **MMLU / FreshQA** benchmarks need their prep files; the old notebook faked an
  "MMLU benchmark" out of five CEO questions (review B1). Get the real files
  before adding those rows, and pass `benchmark="mmlu"` to `evaluate_condition`.